# CS401 — ML & Neural Networks: Foundations

**Lecture 1 — 2026-09-21**

---

## What is Machine Learning?

- **Machine Learning** is a subset of **AI** (Artificial Intelligence)
- Hierarchy: **AI → ML → Neural Networks**
- ML is a branch of AI that learns patterns from data

### Core Pipeline

```
┌─────────┐      ┌──────────────┐      ┌─────────────────┐
│  DATA   │ ───▶ │ ML Algorithm │ ───▶ │ Prediction Model│
└─────────┘      └──────────────┘      └─────────────────┘
```

### Key References

| Reference | Contribution |
|---|---|
| **Andrew Ng** | Prominent ML educator |
| **Tom Mitchell** | *Machine Learning* (textbook) |
| **Jugal Kalita** | *ML Learning Theory and Practice* |
| **Donald Hebb (1949)** | Hebbian learning — foundational to neural networks |

---

## Learning Types

### 1. Supervised Learning

- Uses **labelled** data: features (x) → target (y)
- Tasks: classification, regression
- Example: predicting outcomes from known input-output pairs

### 2. Unsupervised Learning

- Uses **unlabelled** data
- Tasks: clustering, dimensionality reduction
- Example: **customer segmentation** (grouping similar customers)

### 3. Reinforcement Learning (RL)

- Agent takes actions in an environment
- Receives **reward** signals
- Learns a policy to maximise cumulative reward

```
        ┌─────────┐    Action (A₁)    ┌─────────────┐
        │  Agent  │ ───────────────▶  │ Environment │
        └─────────┘                    └─────────────┘
             ▲                              │
             │        State (S₁)            │
             └──────────────────────────────┘
                        ▲
                        │  Reward (R)
```

---

## Interactive Demo: Supervised vs Unsupervised Learning

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Supervised: Regression (house prices from lecture) ---
X_sup = np.array([50, 60, 70, 80, 90, 100, 110, 120, 130, 140])
Y_sup = np.array([150, 180, 210, 240, 270, 300, 330, 360, 390, 420])

axes[0].scatter(X_sup, Y_sup, c='blue', edgecolors='black', zorder=5)
z = np.polyfit(X_sup, Y_sup, 1)
p = np.poly1d(z)
axes[0].plot(X_sup, p(X_sup), 'r--', label=f'y = {z[0]:.1f}x + {z[1]:.0f}')
axes[0].set_xlabel('Area (m²)')
axes[0].set_ylabel('Price (€k)')
axes[0].set_title('Supervised: Regression\n(known labels guide the model)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Supervised: Classification ---
np.random.seed(0)
class0 = np.random.randn(20, 2) + [2, 2]
class1 = np.random.randn(20, 2) + [6, 6]

axes[1].scatter(class0[:, 0], class0[:, 1], c='blue', label='Class 0', edgecolors='black')
axes[1].scatter(class1[:, 0], class1[:, 1], c='red', label='Class 1', edgecolors='black')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_title('Supervised: Classification\n(colours = known labels)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- Unsupervised: Clustering (same data, no labels) ---
all_points = np.vstack([class0, class1])
axes[2].scatter(all_points[:, 0], all_points[:, 1], c='gray', edgecolors='black')
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')
axes[2].set_title('Unsupervised: Clustering\n(no labels — algorithm finds groups)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Reinforcement Learning: Q-Learning Demo

A simple grid-world agent learning to reach a goal via Q-learning.

In [ ]:
class SimpleGridEnv:
    def __init__(self, size=5):
        self.size = size
        self.goal = (size - 1, size - 1)
        self.reset()
    
    def reset(self):
        self.agent_pos = (0, 0)
        return self.agent_pos
    
    def step(self, action):
        """actions: 0=up, 1=down, 2=left, 3=right"""
        x, y = self.agent_pos
        if action == 0: x = max(0, x - 1)
        elif action == 1: x = min(self.size - 1, x + 1)
        elif action == 2: y = max(0, y - 1)
        elif action == 3: y = min(self.size - 1, y + 1)
        
        self.agent_pos = (x, y)
        
        if self.agent_pos == self.goal:
            return self.agent_pos, 10.0, True
        else:
            return self.agent_pos, -0.1, False

# Q-Learning
env = SimpleGridEnv(size=5)
q_table = np.zeros((25, 4))
learning_rate = 0.1
discount_factor = 0.95
episodes = 500
rewards_per_episode = []

for episode in range(episodes):
    state = env.reset()
    state_idx = state[0] * 5 + state[1]
    total_reward = 0
    
    for step in range(100):
        if np.random.random() < 0.1:
            action = np.random.randint(4)
        else:
            action = np.argmax(q_table[state_idx])
        
        next_state, reward, done = env.step(action)
        next_idx = next_state[0] * 5 + next_state[1]
        
        best_next = np.max(q_table[next_idx])
        q_table[state_idx, action] += learning_rate * (
            reward + discount_factor * best_next - q_table[state_idx, action]
        )
        
        state_idx = next_idx
        total_reward += reward
        if done:
            break
    
    rewards_per_episode.append(total_reward)

# Plot learning curve
plt.figure(figsize=(10, 4))
plt.plot(rewards_per_episode, alpha=0.5, label='Episode reward')
window = 20
if len(rewards_per_episode) >= window:
    moving_avg = np.convolve(rewards_per_episode, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(rewards_per_episode)), moving_avg, 'r-', linewidth=2, label=f'{window}-episode moving avg')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('RL Agent Learning Curve\n(Agent learns to reach the goal)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Average reward (first 50 episodes):  {np.mean(rewards_per_episode[:50]):.2f}")
print(f"Average reward (last 50 episodes):   {np.mean(rewards_per_episode[-50:]):.2f}")

---

## ML Tools & Libraries

| Tool | Purpose |
|---|---|
| **NumPy** | Numerical Python — arrays, linear algebra |
| **Pandas** | Tabular data, DataFrames, SQL-like ops |
| **Scikit-Learn** | Classification, regression, clustering, model selection |
| **Matplotlib** | Plotting and visualization |
| **TensorFlow / PyTorch** | Deep learning frameworks |
| **Jupyter Notebook** | Interactive development environment |
| **Anaconda** | Package & environment management |

In [ ]:
# Scikit-Learn quick demo: Linear Regression comparison
from sklearn.linear_model import LinearRegression

X_reshaped = X_sup.reshape(-1, 1)
sklearn_model = LinearRegression()
sklearn_model.fit(X_reshaped, Y_sup)

print("Scikit-Learn LinearRegression:")
print(f"  Intercept (θ₀): {sklearn_model.intercept_:.2f}")
print(f"  Slope (θ₁):     {sklearn_model.coef_[0]:.4f}")
print(f"  R² score:       {sklearn_model.score(X_reshaped, Y_sup):.4f}")
print()
print(f"Prediction for 150 m²: {sklearn_model.predict([[150]])[0]:.2f} €k")

---

## Summary

| Concept | Key Takeaway |
|---|---|
| AI → ML → NN | Neural networks are a subset of ML |
| Supervised | Labelled data, classification & regression |
| Unsupervised | Unlabelled data, clustering & reduction |
| Reinforcement | Reward-based, agent-environment loop |
| Pipeline | DATA → Algorithm → Prediction Model |

### Key Formulas (coming in next lectures)

- **Hypothesis:** $h(x) = \theta_0 + \theta_1 x$
- **MSE Loss:** $J(\theta) = \frac{1}{n} \sum (h(x_i) - y_i)^2$
- **GD Update:** $\theta_j := \theta_j - \alpha \cdot \frac{\partial J}{\partial \theta_j}$
- **Normal Equation:** $\theta = (X^T X)^{-1} X^T y$
- **Sigmoid:** $\sigma(z) = \frac{1}{1 + e^{-z}}$